In [1]:
%matplotlib inline

import os
import sys
import math
import time
import json
import requests

from sys import platform
from pathlib import Path
from functools import partial

sys.path.append('../../')

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from importlib.metadata import version 

import tiktoken
import numpy as np
import tensorflow

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

%load_ext autoreload
%autoreload 2

from llm_from_scratch.CH4.gpt import GPTModel, generate_text_simple
from llm_from_scratch.CH5.utils import load_weights_into_gpt, text_to_token_ids, token_ids_to_text, generate, plot_losses
from llm_from_scratch.CH5.gpt_download import download_and_load_gpt2
from llm_from_scratch.CH5.loss import calc_loss_batch, calc_loss_loader
from llm_from_scratch.CH5.optim import train_model

# from spam_dataset import SpamDataset, download_and_unzip_spam_data, create_balanced_dataset, random_split
# from utils import calc_accuracy_loader, calc_loss_loader, calc_loss_batch, train_classifier
from llm_from_scratch.CH6.optim import create_optimizer, LayerDecayValueAssigner

from utils import check_if_running, query_model, generate_model_scores
from instruction_data import download_and_load_file, format_input, InstructionDataset, custom_collate_fn

In [2]:
ollama_running=check_if_running('ollama')
if not ollama_running: raise RuntimeError("Ollama not running. Launch ollama before proceeding")
print("Ollama running: ", check_if_running('ollama'))

Ollama running:  True


In [3]:
main_dirpath="/Users/reaungamornrat.sureerat/data/instruction_finetuning"
file_path=f"{main_dirpath}/instruction-data-with-response.json"
with open(file_path, 'r') as file: test_data=json.load(file)

In [4]:
model='llama3'
result=query_model("What do Llamas eat?", model)
print(result)

Llamas are herbivores, which means they primarily feed on plant-based foods. Their diet typically consists of:

1. Grasses: Llamas love to graze on various types of grasses, including tall grasses, short grasses, and even weeds.
2. Hay: High-quality hay, such as alfalfa or timothy hay, is a staple in a llama's diet. They enjoy the sweet taste and texture of fresh hay.
3. Grains: Llamas may receive grains like oats, barley, or corn as part of their daily ration. However, it's essential to provide these grains in moderation, as they can be high in calories.
4. Fruits and vegetables: Llamas enjoy a variety of fruits and veggies, such as apples, carrots, sweet potatoes, and leafy greens like kale or spinach.
5. Minerals: Llamas require access to mineral supplements, which help maintain their overall health and well-being.

In the wild, llamas might also eat:

1. Leaves: They'll munch on leaves from trees and shrubs, including plants like willow, alder, and birch.
2. Bark: In some cases, ll

In [5]:
for entry in test_data[:3]:
    prompt=(
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score."
    )
    print("\nDataset response: ")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry['model_response'])
    print("\nScore: ")
    print(">>", query_model(prompt))
    print("\n", "-"*20)


Dataset response: 
>> The car is as fast as lightning.

Model response:
>>The car is as fast as a cheetah.

Score: 
>> I'd rate the model response "The car is as fast as a cheetah." an 85 out of 100.

Here's why:

* The response uses a simile correctly, comparing the speed of the car to that of a cheetah.
* The comparison is relevant and makes sense, as both cars and cheetahs are known for their speed.
* The phrase "as fast as" is used correctly to introduce the simile.

However, I wouldn't give it a perfect score because:

* While a cheetah is indeed very fast, it's not necessarily the most iconic or universally recognized example of speed. Lightning, on the other hand, is often used as a metaphor for incredible speed.
* The response could be improved by using more vivid language to describe the car's speed. For example, "The car is as swift as a cheetah" or "The car zooms along like a cheetah on the prowl."

Overall, the model response is a good effort, but it could be even stronger

In [6]:
# A longer improved prompt, employing a grading rubic, resulting in more accurate and less noisy evaluations
instruction=format_input(entry)
reference=entry['output']
answer=entry['model_response']

prompt=f"""
You are a fair judge assistant tasked with providing clear, objective feedback based on specific criteria, ensuring each assessment reflects the absolute standards set for performance.
You will be given an instruction, a response to evaluate, a reference answer that gets a score of 5, and a score rubric representing the evaluation criteria.
Write a detailed feedback that assesses the qulaity of the response strictly based on the given score rubric, not evaluating in general.
Please do not generate any other opening, closing, and explanations.

Here is the rubric you should use to build your answer:
1: The response fails to address the instructions, providing irrelevant, incorrect, or excessively verbose information that detracts from the user's request.
2: The response partially addresses the instructions but includes significant inaccuracies, irrelevant details, or excessive elaboration that detracts from the main task.
3: The response follows the instructions with some minor inaccuracies or omissions. It is generally relevent and clear ,but may include some unnecessary details or could be more concise.
4: The response adheres to the instructions, offering clear, accurate, and relevant information in a concise manner, with only occasional, minor instances of excessive detail or slight lack of clarity.
5: The response fully adheres to the instructions, providing a clear, accurate, and relevant answer in a concise and efficient manner. It addresses all aspects of the request without unnecessary details or elaboration.

Provide your feedback as follows:

Feedback:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here is the instruction, the reference answer, and the response.

Instruction: {instruction}
Reference Answer: {reference}
Answer: {answer}

Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.
Feedback:::
Evaluation:"""

query_model(prompt)

"Feedback:::\nThe response fails to address the instructions by providing an incorrect author for 'Pride and Prejudice'. The reference answer clearly states that the author is Jane Austen, but the response incorrectly identifies George Bernard Shaw as the author. This significant inaccuracy detracts from the main task of naming the correct author.\n\nTotal rating: 1"

In [7]:
scores=generate_model_scores(test_data, "model_response")
print(f"Numver of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}")

# Not masking instruction
# Numver of scores: 110 of 110
# Average score: 49.80

Numver of scores: 110 of 110
Average score: 50.15
